In [5]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold
from sklearn.base import clone
from sklearn.preprocessing import MaxAbsScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
)

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

RANDOM_STATE = 0
CV_SPLITS = 5

def mean_std(x): return f"{np.mean(x):.3f} ± {np.std(x):.3f}"

def cv_eval_with_confusion(model, X, y, cv):
    accs, precs, recs, f1s, aucs = [], [], [], [], []
    y_true_all, y_pred_all = [], []

    for tr_idx, te_idx in cv.split(X, y):
        m = clone(model)
        X_tr, X_te = X.iloc[tr_idx], X.iloc[te_idx]
        y_tr, y_te = y.iloc[tr_idx], y.iloc[te_idx]

        m.fit(X_tr, y_tr)
        y_pred = m.predict(X_te)

        y_true_all.append(y_te.to_numpy())
        y_pred_all.append(y_pred)

        accs.append(accuracy_score(y_te, y_pred))
        precs.append(precision_score(y_te, y_pred, pos_label=1, zero_division=0))
        recs.append(recall_score(y_te, y_pred, pos_label=1, zero_division=0))
        f1s.append(f1_score(y_te, y_pred, pos_label=1, zero_division=0))

        # ROC-AUC needs probabilities
        proba = m.predict_proba(X_te)[:, 1]
        aucs.append(roc_auc_score(y_te, proba))

    y_true_all = np.concatenate(y_true_all)
    y_pred_all = np.concatenate(y_pred_all)
    cm = confusion_matrix(y_true_all, y_pred_all, labels=[0, 1])

    metrics = {
        "accuracy": mean_std(accs),
        "precision_1": mean_std(precs),
        "recall_1": mean_std(recs),
        "f1_1": mean_std(f1s),
        "roc_auc": mean_std(aucs),
    }
    return metrics, cm

# Load
df = pd.read_csv("diagnosed_diabetes for classification.csv")
y = df["diagnosed_diabetes"].astype(int)
X = df.drop(columns=["diagnosed_diabetes"])

print("Class counts:\n", y.value_counts())

cv = StratifiedKFold(n_splits=CV_SPLITS, shuffle=True, random_state=RANDOM_STATE)

pipe_unbalanced = Pipeline([
    ("scaler", MaxAbsScaler()),
    ("lr", LogisticRegression(solver="saga", max_iter=3000, random_state=RANDOM_STATE))
])

pipe_class_weight = Pipeline([
    ("scaler", MaxAbsScaler()),
    ("lr", LogisticRegression(solver="saga", max_iter=3000, random_state=RANDOM_STATE, class_weight="balanced"))
])

pipe_smote = ImbPipeline([
    ("scaler", MaxAbsScaler()),
    ("smote", SMOTE(random_state=RANDOM_STATE)),
    ("lr", LogisticRegression(solver="saga", max_iter=3000, random_state=RANDOM_STATE))
])

for name, model in [
    ("UNBALANCED", pipe_unbalanced),
    ("BALANCED (class_weight)", pipe_class_weight),
    ("BALANCED (SMOTE)", pipe_smote),
]:
    metrics, cm = cv_eval_with_confusion(model, X, y, cv)
    print(f"\n=== {name} ===")
    for k, v in metrics.items():
        print(f"{k:>10}: {v}")
    print("Confusion matrix (OOF, aggregated across CV folds):")
    print(pd.DataFrame(cm, index=["true_0","true_1"], columns=["pred_0","pred_1"]))


Class counts:
 diagnosed_diabetes
1    59998
0    40002
Name: count, dtype: int64

=== UNBALANCED ===
  accuracy: 0.632 ± 0.003
precision_1: 0.654 ± 0.002
  recall_1: 0.824 ± 0.002
      f1_1: 0.729 ± 0.002
   roc_auc: 0.660 ± 0.004
Confusion matrix (OOF, aggregated across CV folds):
        pred_0  pred_1
true_0   13782   26220
true_1   10544   49454

=== BALANCED (class_weight) ===
  accuracy: 0.602 ± 0.001
precision_1: 0.716 ± 0.003
  recall_1: 0.557 ± 0.002
      f1_1: 0.627 ± 0.001
   roc_auc: 0.660 ± 0.004
Confusion matrix (OOF, aggregated across CV folds):
        pred_0  pred_1
true_0   26779   13223
true_1   26587   33411

=== BALANCED (SMOTE) ===
  accuracy: 0.602 ± 0.001
precision_1: 0.716 ± 0.002
  recall_1: 0.559 ± 0.002
      f1_1: 0.628 ± 0.001
   roc_auc: 0.659 ± 0.004
Confusion matrix (OOF, aggregated across CV folds):
        pred_0  pred_1
true_0   26656   13346
true_1   26434   33564


In [9]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold
from sklearn.base import clone
from sklearn.preprocessing import MaxAbsScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, accuracy_score, f1_score, recall_score

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

RANDOM_STATE = 0
CV_SPLITS = 5

def mean_std(x): return f"{np.mean(x):.3f} ± {np.std(x):.3f}"

def cv_eval_with_confusion(model, X, y, cv, multiclass=False):
    accs, f1s, recs = [], [], []
    y_true_all, y_pred_all = [], []

    labels = sorted(pd.unique(y))

    for tr_idx, te_idx in cv.split(X, y):
        m = clone(model)
        X_tr, X_te = X.iloc[tr_idx], X.iloc[te_idx]
        y_tr, y_te = y.iloc[tr_idx], y.iloc[te_idx]

        m.fit(X_tr, y_tr)
        y_pred = m.predict(X_te)

        y_true_all.append(y_te.to_numpy())
        y_pred_all.append(y_pred)

        accs.append(accuracy_score(y_te, y_pred))
        if multiclass:
            f1s.append(f1_score(y_te, y_pred, average="macro", zero_division=0))
            recs.append(recall_score(y_te, y_pred, average="macro", zero_division=0))
        else:
            f1s.append(f1_score(y_te, y_pred, pos_label=1, zero_division=0))
            recs.append(recall_score(y_te, y_pred, pos_label=1, zero_division=0))

    y_true_all = np.concatenate(y_true_all)
    y_pred_all = np.concatenate(y_pred_all)
    cm = confusion_matrix(y_true_all, y_pred_all, labels=labels)

    metrics = {"accuracy": mean_std(accs)}
    if multiclass:
        metrics["f1_macro"] = mean_std(f1s)
        metrics["recall_macro"] = mean_std(recs)
    else:
        metrics["f1_1"] = mean_std(f1s)
        metrics["recall_1"] = mean_std(recs)

    return metrics, cm, labels

# Load
df = pd.read_csv("prediabetes_for_classification.csv")

target = "diabetes_stage"
y = df[target]
if y.nunique() <= 20:
    y = y.astype(int)
X = df.drop(columns=[target])

print("Class counts:\n", y.value_counts())

n_classes = y.nunique()
is_multiclass = (n_classes > 2)

cv = StratifiedKFold(n_splits=CV_SPLITS, shuffle=True, random_state=RANDOM_STATE)

lr_kwargs = dict(solver="saga", max_iter=3000, random_state=RANDOM_STATE)
if is_multiclass:
    lr_kwargs["multi_class"] = "multinomial"

pipe_unbalanced = Pipeline([
    ("scaler", MaxAbsScaler()),
    ("lr", LogisticRegression(**lr_kwargs))
])

pipe_class_weight = Pipeline([
    ("scaler", MaxAbsScaler()),
    ("lr", LogisticRegression(**lr_kwargs, class_weight="balanced"))
])

pipe_smote = ImbPipeline([
    ("scaler", MaxAbsScaler()),
    ("smote", SMOTE(random_state=RANDOM_STATE)),
    ("lr", LogisticRegression(**lr_kwargs))
])

for name, model in [
    ("UNBALANCED", pipe_unbalanced),
    ("BALANCED (class_weight)", pipe_class_weight),
    ("BALANCED (SMOTE)", pipe_smote),
]:
    metrics, cm, labels = cv_eval_with_confusion(model, X, y, cv, multiclass=is_multiclass)
    print(f"\n=== {name} ===")
    for k, v in metrics.items():
        print(f"{k:>12}: {v}")
    print("Confusion matrix (OOF, aggregated across CV folds):")
    cm_df = pd.DataFrame(cm, index=[f"true_{l}" for l in labels], columns=[f"pred_{l}" for l in labels])
    print(cm_df)


Class counts:
 diabetes_stage
1    31845
0     7981
Name: count, dtype: int64

=== UNBALANCED ===
    accuracy: 0.799 ± 0.000
        f1_1: 0.888 ± 0.000
    recall_1: 0.999 ± 0.000
Confusion matrix (OOF, aggregated across CV folds):
        pred_0  pred_1
true_0      14    7967
true_1      21   31824

=== BALANCED (class_weight) ===
    accuracy: 0.576 ± 0.005
        f1_1: 0.678 ± 0.005
    recall_1: 0.560 ± 0.007
Confusion matrix (OOF, aggregated across CV folds):
        pred_0  pred_1
true_0    5107    2874
true_1   14027   17818

=== BALANCED (SMOTE) ===
    accuracy: 0.575 ± 0.002
        f1_1: 0.678 ± 0.002
    recall_1: 0.560 ± 0.003
Confusion matrix (OOF, aggregated across CV folds):
        pred_0  pred_1
true_0    5069    2912
true_1   14000   17845


In [8]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold
from sklearn.base import clone
from sklearn.preprocessing import MaxAbsScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
)

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

RANDOM_STATE = 0
CV_SPLITS = 5
TOP_Q = 0.80  # top 20% as "high risk" (change to 0.90 for top 10%)

def mean_std(x): return f"{np.mean(x):.3f} ± {np.std(x):.3f}"

def cv_eval_with_confusion(model, X, y, cv):
    accs, precs, recs, f1s, aucs = [], [], [], [], []
    y_true_all, y_pred_all = [], []

    for tr_idx, te_idx in cv.split(X, y):
        m = clone(model)
        X_tr, X_te = X.iloc[tr_idx], X.iloc[te_idx]
        y_tr, y_te = y.iloc[tr_idx], y.iloc[te_idx]

        m.fit(X_tr, y_tr)
        y_pred = m.predict(X_te)

        y_true_all.append(y_te.to_numpy())
        y_pred_all.append(y_pred)

        accs.append(accuracy_score(y_te, y_pred))
        precs.append(precision_score(y_te, y_pred, pos_label=1, zero_division=0))
        recs.append(recall_score(y_te, y_pred, pos_label=1, zero_division=0))
        f1s.append(f1_score(y_te, y_pred, pos_label=1, zero_division=0))

        proba = m.predict_proba(X_te)[:, 1]
        aucs.append(roc_auc_score(y_te, proba))

    y_true_all = np.concatenate(y_true_all)
    y_pred_all = np.concatenate(y_pred_all)
    cm = confusion_matrix(y_true_all, y_pred_all, labels=[0, 1])

    metrics = {
        "accuracy": mean_std(accs),
        "precision_1": mean_std(precs),
        "recall_1": mean_std(recs),
        "f1_1": mean_std(f1s),
        "roc_auc": mean_std(aucs),
    }
    return metrics, cm

# Load + build binary target
df = pd.read_csv("diabetes_preprocessed_for_risk_score.csv")
thr = df["diabetes_risk_score"].quantile(TOP_Q)

df = df.copy()
df["risk_high"] = (df["diabetes_risk_score"] >= thr).astype(int)
df = df.drop(columns=["diabetes_risk_score"])

y = df["risk_high"].astype(int)
X = df.drop(columns=["risk_high"])

print(f"High-risk threshold (q={TOP_Q}): {thr:.3f}")
print("Class counts:\n", y.value_counts())

cv = StratifiedKFold(n_splits=CV_SPLITS, shuffle=True, random_state=RANDOM_STATE)

pipe_unbalanced = Pipeline([
    ("scaler", MaxAbsScaler()),
    ("lr", LogisticRegression(solver="saga", max_iter=3000, random_state=RANDOM_STATE))
])

pipe_class_weight = Pipeline([
    ("scaler", MaxAbsScaler()),
    ("lr", LogisticRegression(solver="saga", max_iter=3000, random_state=RANDOM_STATE, class_weight="balanced"))
])

pipe_smote = ImbPipeline([
    ("scaler", MaxAbsScaler()),
    ("smote", SMOTE(random_state=RANDOM_STATE)),
    ("lr", LogisticRegression(solver="saga", max_iter=3000, random_state=RANDOM_STATE))
])

for name, model in [
    ("UNBALANCED", pipe_unbalanced),
    ("BALANCED (class_weight)", pipe_class_weight),
    ("BALANCED (SMOTE)", pipe_smote),
]:
    metrics, cm = cv_eval_with_confusion(model, X, y, cv)
    print(f"\n=== {name} ===")
    for k, v in metrics.items():
        print(f"{k:>10}: {v}")
    print("Confusion matrix (OOF, aggregated across CV folds):")
    print(pd.DataFrame(cm, index=["true_0","true_1"], columns=["pred_0","pred_1"]))


High-risk threshold (q=0.8): 37.700
Class counts:
 risk_high
0    79787
1    20213
Name: count, dtype: int64

=== UNBALANCED ===
  accuracy: 0.980 ± 0.001
precision_1: 0.958 ± 0.002
  recall_1: 0.941 ± 0.004
      f1_1: 0.949 ± 0.002
   roc_auc: 0.998 ± 0.000
Confusion matrix (OOF, aggregated across CV folds):
        pred_0  pred_1
true_0   78943     844
true_1    1194   19019

=== BALANCED (class_weight) ===
  accuracy: 0.969 ± 0.001
precision_1: 0.882 ± 0.003
  recall_1: 0.980 ± 0.002
      f1_1: 0.928 ± 0.001
   roc_auc: 0.998 ± 0.000
Confusion matrix (OOF, aggregated across CV folds):
        pred_0  pred_1
true_0   77125    2662
true_1     412   19801

=== BALANCED (SMOTE) ===
  accuracy: 0.974 ± 0.001
precision_1: 0.906 ± 0.003
  recall_1: 0.971 ± 0.002
      f1_1: 0.937 ± 0.002
   roc_auc: 0.998 ± 0.000
Confusion matrix (OOF, aggregated across CV folds):
        pred_0  pred_1
true_0   77742    2045
true_1     582   19631


In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import KFold, cross_validate
from sklearn.linear_model import LinearRegression

# Load encoded dataset for risk score regression
df = pd.read_csv("diabetes_preprocessed_for_risk_score.csv")

target = "diabetes_risk_score"

# Drop potential leakage targets if they exist
leak_cols = [c for c in ["diagnosed_diabetes", "diabetes_stage"] if c in df.columns]

X = df.drop(columns=[target] + leak_cols)
y = df[target]

cv = KFold(n_splits=5, shuffle=True, random_state=0)

model = LinearRegression()

scoring = {
    "r2": "r2",
    "mae": "neg_mean_absolute_error",
    "rmse": "neg_root_mean_squared_error",
}

scores = cross_validate(model, X, y, cv=cv, scoring=scoring, n_jobs=-1)

r2 = scores["test_r2"]
mae = -scores["test_mae"]
rmse = -scores["test_rmse"]

print(f"R2:   {r2.mean():.3f} ± {r2.std():.3f}")
print(f"MAE:  {mae.mean():.3f} ± {mae.std():.3f}")
print(f"RMSE: {rmse.mean():.3f} ± {rmse.std():.3f}")


R2:   0.979 ± 0.001
MAE:  1.002 ± 0.005
RMSE: 1.305 ± 0.016


In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import KFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor

# Load encoded risk-score dataset
df = pd.read_csv("diabetes_preprocessed_for_risk_score.csv")

target = "diabetes_risk_score"

# Avoid leakage if other targets exist
leak_cols = [c for c in ["diagnosed_diabetes", "diabetes_stage"] if c in df.columns]
X = df.drop(columns=[target] + leak_cols)
y = df[target]

cv = KFold(n_splits=5, shuffle=True, random_state=0)

models = {
    "dummy_mean": DummyRegressor(strategy="mean"),
    "ridge": Pipeline([("scaler", StandardScaler()), ("m", Ridge(alpha=1.0))]),
    "elasticnet": Pipeline([("scaler", StandardScaler()), ("m", ElasticNet(alpha=0.01, l1_ratio=0.5, random_state=0))]),
    "random_forest": RandomForestRegressor(n_estimators=300, random_state=0, n_jobs=-1),
    "hist_gb": HistGradientBoostingRegressor(random_state=0),
}

scoring = {
    "r2": "r2",
    "mae": "neg_mean_absolute_error",
    "rmse": "neg_root_mean_squared_error",
}

for name, model in models.items():
    scores = cross_validate(model, X, y, cv=cv, scoring=scoring, n_jobs=-1)
    r2 = scores["test_r2"]
    mae = -scores["test_mae"]
    rmse = -scores["test_rmse"]
    print(f"\n=== {name} ===")
    print(f"R2:   {r2.mean():.3f} ± {r2.std():.3f}")
    print(f"MAE:  {mae.mean():.3f} ± {mae.std():.3f}")
    print(f"RMSE: {rmse.mean():.3f} ± {rmse.std():.3f}")



=== dummy_mean ===
R2:   -0.000 ± 0.000
MAE:  7.213 ± 0.017
RMSE: 9.062 ± 0.017

=== ridge ===
R2:   0.979 ± 0.001
MAE:  1.002 ± 0.005
RMSE: 1.305 ± 0.016

=== elasticnet ===
R2:   0.979 ± 0.000
MAE:  1.005 ± 0.004
RMSE: 1.306 ± 0.016
